In [0]:
from pyspark.sql import functions as F


# ============================================================
# 1. Load existing customer-product feature table
# ============================================================

reorder_features_df = spark.table(
    "workspace.ml_data.reorder_features"
)


print("Feature table loaded successfully.")
print("Rows:", reorder_features_df.count())
print(
    "Customers:",
    reorder_features_df
    .select("user_id")
    .distinct()
    .count()
)

print()
print("Available columns:")
print("=" * 60)

for column in reorder_features_df.columns:
    print(column)

Feature table loaded successfully.
Rows: 8474661
Customers: 131209

Available columns:
user_id
target_order_id
product_id
target_reordered
customer_prior_orders
customer_total_products
customer_unique_products
customer_avg_basket_size
customer_reorder_rate
customer_avg_days_between_orders
customer_std_days_between_orders
customer_avg_order_hour
customer_active_days_of_week
customer_last_basket_size
customer_avg_last_3_basket_size
customer_basket_size_trend
customer_preferred_dow
customer_preferred_day_share
customer_preferred_hour
customer_preferred_hour_share
product_purchase_count
product_unique_customers
product_reorder_rate
product_avg_cart_position
product_name
aisle_id
department_id
product_purchases_per_customer
product_order_share
product_preferred_dow
product_preferred_day_share
product_preferred_hour
product_preferred_hour_share
user_product_order_count
user_product_reorder_rate
user_product_avg_cart_position
user_product_first_order
user_product_last_order
user_product_order

In [0]:
from pyspark.sql import functions as F


# ============================================================
# 2. Build one Shopping DNA profile per customer
# ============================================================

customer_dna_df = (
    reorder_features_df
    .select(
        "user_id",

        # Customer history
        "customer_prior_orders",
        "customer_total_products",
        "customer_unique_products",

        # Basket behavior
        "customer_avg_basket_size",
        "customer_last_basket_size",
        "customer_avg_last_3_basket_size",
        "customer_basket_size_trend",

        # Loyalty / reorder behavior
        "customer_reorder_rate",

        # Shopping rhythm
        "customer_avg_days_between_orders",
        "customer_std_days_between_orders",

        # Time behavior
        "customer_avg_order_hour",
        "customer_active_days_of_week",
        "customer_preferred_dow",
        "customer_preferred_day_share",
        "customer_preferred_hour",
        "customer_preferred_hour_share",
    )
    .dropDuplicates(["user_id"])
)


# ============================================================
# 3. Add human-readable Shopping DNA signals
# ============================================================

customer_dna_df = (
    customer_dna_df

    # --------------------------------------------------------
    # Reorder loyalty
    # --------------------------------------------------------
    .withColumn(
        "reorder_loyalty_pct",
        F.round(
            F.col("customer_reorder_rate") * 100,
            1,
        ),
    )

    .withColumn(
        "loyalty_profile",
        F.when(
            F.col("customer_reorder_rate") >= 0.70,
            "HIGH LOYALTY",
        )
        .when(
            F.col("customer_reorder_rate") >= 0.45,
            "MODERATE LOYALTY",
        )
        .otherwise(
            "EXPLORATORY",
        ),
    )

    # --------------------------------------------------------
    # Shopping frequency
    # --------------------------------------------------------
    .withColumn(
        "shopping_frequency",
        F.when(
            F.col("customer_avg_days_between_orders") <= 7,
            "FREQUENT",
        )
        .when(
            F.col("customer_avg_days_between_orders") <= 14,
            "REGULAR",
        )
        .otherwise(
            "OCCASIONAL",
        ),
    )

    # --------------------------------------------------------
    # Basket momentum
    # --------------------------------------------------------
    .withColumn(
        "basket_momentum",
        F.when(
            F.col("customer_basket_size_trend") > 1,
            "GROWING",
        )
        .when(
            F.col("customer_basket_size_trend") < -1,
            "SHRINKING",
        )
        .otherwise(
            "STABLE",
        ),
    )

    # --------------------------------------------------------
    # Order regularity
    #
    # Lower variation in days between orders =
    # more predictable shopping rhythm.
    # --------------------------------------------------------
    .withColumn(
        "shopping_regularity",
        F.when(
            F.col("customer_std_days_between_orders") <= 4,
            "HIGH",
        )
        .when(
            F.col("customer_std_days_between_orders") <= 9,
            "MODERATE",
        )
        .otherwise(
            "FLEXIBLE",
        ),
    )

    # --------------------------------------------------------
    # Preferred shopping time
    # --------------------------------------------------------
    .withColumn(
        "preferred_time_period",
        F.when(
            F.col("customer_preferred_hour") < 6,
            "LATE NIGHT",
        )
        .when(
            F.col("customer_preferred_hour") < 12,
            "MORNING",
        )
        .when(
            F.col("customer_preferred_hour") < 17,
            "AFTERNOON",
        )
        .when(
            F.col("customer_preferred_hour") < 21,
            "EVENING",
        )
        .otherwise(
            "NIGHT",
        ),
    )

    # --------------------------------------------------------
    # Customer persona
    # --------------------------------------------------------
    .withColumn(
        "shopping_persona",
        F.when(
            (F.col("customer_reorder_rate") >= 0.70)
            &
            (F.col("customer_avg_days_between_orders") <= 14),
            "LOYAL REPLENISHER",
        )
        .when(
            (F.col("customer_reorder_rate") < 0.45)
            &
            (F.col("customer_unique_products") >= 50),
            "CURIOUS EXPLORER",
        )
        .when(
            F.col("customer_avg_days_between_orders") <= 7,
            "FREQUENT PLANNER",
        )
        .when(
            F.col("customer_basket_size_trend") > 1,
            "GROWING BASKET SHOPPER",
        )
        .otherwise(
            "BALANCED SHOPPER",
        ),
    )
)


# ============================================================
# 4. Final UI-friendly fields
# ============================================================

customer_dna_df = (
    customer_dna_df
    .withColumn(
        "avg_basket_size",
        F.round(
            F.col("customer_avg_basket_size"),
            1,
        ),
    )
    .withColumn(
        "avg_days_between_orders",
        F.round(
            F.col("customer_avg_days_between_orders"),
            1,
        ),
    )
    .withColumn(
        "preferred_hour",
        F.col("customer_preferred_hour").cast("int"),
    )
)


# ============================================================
# 5. Validation
# ============================================================

print("Customer Shopping DNA dataset created.")
print("Rows:", customer_dna_df.count())

print(
    "Customers:",
    customer_dna_df
    .select("user_id")
    .distinct()
    .count()
)


display(
    customer_dna_df
    .select(
        "user_id",
        "shopping_persona",
        "reorder_loyalty_pct",
        "loyalty_profile",
        "shopping_frequency",
        "basket_momentum",
        "shopping_regularity",
        "avg_basket_size",
        "avg_days_between_orders",
        "preferred_hour",
        "preferred_time_period",
        "customer_prior_orders",
        "customer_unique_products",
    )
    .orderBy("user_id")
    .limit(30)
)

Customer Shopping DNA dataset created.
Rows: 131209
Customers: 131209


user_id,shopping_persona,reorder_loyalty_pct,loyalty_profile,shopping_frequency,basket_momentum,shopping_regularity,avg_basket_size,avg_days_between_orders,preferred_hour,preferred_time_period,customer_prior_orders,customer_unique_products
1,GROWING BASKET SHOPPER,69.5,MODERATE LOYALTY,OCCASIONAL,GROWING,MODERATE,5.9,19.6,7,MORNING,10,18
2,BALANCED SHOPPER,47.7,MODERATE LOYALTY,OCCASIONAL,STABLE,FLEXIBLE,13.9,15.2,10,MORNING,14,102
5,BALANCED SHOPPER,37.8,EXPLORATORY,REGULAR,STABLE,MODERATE,9.3,13.3,18,EVENING,4,23
7,BALANCED SHOPPER,67.0,MODERATE LOYALTY,REGULAR,STABLE,MODERATE,10.3,10.7,9,MORNING,20,68
8,BALANCED SHOPPER,26.5,EXPLORATORY,OCCASIONAL,STABLE,HIGH,16.3,30.0,0,LATE NIGHT,3,36
9,CURIOUS EXPLORER,23.7,EXPLORATORY,OCCASIONAL,STABLE,FLEXIBLE,25.3,18.0,12,AFTERNOON,3,58
10,CURIOUS EXPLORER,34.3,EXPLORATORY,OCCASIONAL,GROWING,MODERATE,28.6,19.8,15,AFTERNOON,5,94
13,GROWING BASKET SHOPPER,64.2,MODERATE LOYALTY,REGULAR,GROWING,HIGH,6.8,7.6,12,AFTERNOON,12,29
14,CURIOUS EXPLORER,32.4,EXPLORATORY,OCCASIONAL,SHRINKING,FLEXIBLE,16.2,22.1,8,MORNING,13,142
17,LOYAL REPLENISHER,71.8,HIGH LOYALTY,REGULAR,GROWING,MODERATE,7.4,7.4,12,AFTERNOON,40,83


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 3. CUSTOMER CATEGORY SIGNATURE
#
# Identify each customer's strongest departments and aisles
# using the affinity features already engineered in Notebook 07.
# ============================================================


# ============================================================
# 3.1 Load category reference tables
# ============================================================

departments_df = (
    spark.table("workspace.cleaned_data.departments")
    .select(
        "department_id",
        "department",
    )
)

aisles_df = (
    spark.table("workspace.cleaned_data.aisles")
    .select(
        "aisle_id",
        "aisle",
    )
)


print("Category reference tables loaded.")


# ============================================================
# 3.2 Customer × Department affinity
#
# The same department affinity may appear on multiple candidate
# products, so collapse it to one row per customer/department.
# ============================================================

customer_department_signature_df = (
    reorder_features_df
    .groupBy(
        "user_id",
        "department_id",
    )
    .agg(
        F.max(
            "customer_department_affinity"
        ).alias(
            "department_affinity"
        )
    )
    .join(
        departments_df,
        on="department_id",
        how="left",
    )
    .filter(
        F.col("department").isNotNull()
        &
        (F.lower(F.col("department")) != "missing")
    )
)


# ============================================================
# 3.3 Rank departments inside each customer
# ============================================================

department_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("department_affinity"),
        F.asc("department_id"),
    )
)


customer_department_ranked_df = (
    customer_department_signature_df
    .withColumn(
        "department_rank",
        F.row_number().over(
            department_window
        ),
    )
)


# Keep only top 3
top_departments_df = (
    customer_department_ranked_df
    .filter(
        F.col("department_rank") <= 3
    )
)


# ============================================================
# 3.4 Convert top departments to one customer row
# ============================================================

top_departments_wide_df = (
    top_departments_df
    .groupBy("user_id")
    .agg(

        # ----------------------------------------------------
        # Favorite department
        # ----------------------------------------------------

        F.first(
            F.when(
                F.col("department_rank") == 1,
                F.col("department"),
            ),
            ignorenulls=True,
        ).alias(
            "favorite_department"
        ),

        F.first(
            F.when(
                F.col("department_rank") == 1,
                F.round(
                    F.col("department_affinity") * 100,
                    1,
                ),
            ),
            ignorenulls=True,
        ).alias(
            "favorite_department_affinity_pct"
        ),

        # ----------------------------------------------------
        # Second department
        # ----------------------------------------------------

        F.first(
            F.when(
                F.col("department_rank") == 2,
                F.col("department"),
            ),
            ignorenulls=True,
        ).alias(
            "second_department"
        ),

        F.first(
            F.when(
                F.col("department_rank") == 2,
                F.round(
                    F.col("department_affinity") * 100,
                    1,
                ),
            ),
            ignorenulls=True,
        ).alias(
            "second_department_affinity_pct"
        ),

        # ----------------------------------------------------
        # Third department
        # ----------------------------------------------------

        F.first(
            F.when(
                F.col("department_rank") == 3,
                F.col("department"),
            ),
            ignorenulls=True,
        ).alias(
            "third_department"
        ),

        F.first(
            F.when(
                F.col("department_rank") == 3,
                F.round(
                    F.col("department_affinity") * 100,
                    1,
                ),
            ),
            ignorenulls=True,
        ).alias(
            "third_department_affinity_pct"
        ),
    )
)


# ============================================================
# 3.5 Customer × Aisle affinity
# ============================================================

customer_aisle_signature_df = (
    reorder_features_df
    .groupBy(
        "user_id",
        "aisle_id",
    )
    .agg(
        F.max(
            "customer_aisle_affinity"
        ).alias(
            "aisle_affinity"
        )
    )
    .join(
        aisles_df,
        on="aisle_id",
        how="left",
    )
    .filter(
        F.col("aisle").isNotNull()
        &
        (F.lower(F.col("aisle")) != "missing")
    )
)


# ============================================================
# 3.6 Rank aisles inside each customer
# ============================================================

aisle_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("aisle_affinity"),
        F.asc("aisle_id"),
    )
)


customer_aisle_ranked_df = (
    customer_aisle_signature_df
    .withColumn(
        "aisle_rank",
        F.row_number().over(
            aisle_window
        ),
    )
)


top_aisles_df = (
    customer_aisle_ranked_df
    .filter(
        F.col("aisle_rank") <= 3
    )
)


# ============================================================
# 3.7 Convert top aisles to one customer row
# ============================================================

top_aisles_wide_df = (
    top_aisles_df
    .groupBy("user_id")
    .agg(

        # ----------------------------------------------------
        # Favorite aisle
        # ----------------------------------------------------

        F.first(
            F.when(
                F.col("aisle_rank") == 1,
                F.col("aisle"),
            ),
            ignorenulls=True,
        ).alias(
            "favorite_aisle"
        ),

        F.first(
            F.when(
                F.col("aisle_rank") == 1,
                F.round(
                    F.col("aisle_affinity") * 100,
                    1,
                ),
            ),
            ignorenulls=True,
        ).alias(
            "favorite_aisle_affinity_pct"
        ),

        # ----------------------------------------------------
        # Second aisle
        # ----------------------------------------------------

        F.first(
            F.when(
                F.col("aisle_rank") == 2,
                F.col("aisle"),
            ),
            ignorenulls=True,
        ).alias(
            "second_aisle"
        ),

        F.first(
            F.when(
                F.col("aisle_rank") == 2,
                F.round(
                    F.col("aisle_affinity") * 100,
                    1,
                ),
            ),
            ignorenulls=True,
        ).alias(
            "second_aisle_affinity_pct"
        ),

        # ----------------------------------------------------
        # Third aisle
        # ----------------------------------------------------

        F.first(
            F.when(
                F.col("aisle_rank") == 3,
                F.col("aisle"),
            ),
            ignorenulls=True,
        ).alias(
            "third_aisle"
        ),

        F.first(
            F.when(
                F.col("aisle_rank") == 3,
                F.round(
                    F.col("aisle_affinity") * 100,
                    1,
                ),
            ),
            ignorenulls=True,
        ).alias(
            "third_aisle_affinity_pct"
        ),
    )
)


# ============================================================
# 3.8 Join category signature into Shopping DNA
# ============================================================

customer_dna_enriched_df = (
    customer_dna_df
    .join(
        top_departments_wide_df,
        on="user_id",
        how="left",
    )
    .join(
        top_aisles_wide_df,
        on="user_id",
        how="left",
    )
)


# ============================================================
# 3.9 Validation
# ============================================================

dna_rows = customer_dna_enriched_df.count()

dna_customers = (
    customer_dna_enriched_df
    .select("user_id")
    .distinct()
    .count()
)

duplicate_customers = (
    customer_dna_enriched_df
    .groupBy("user_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


print()
print("CUSTOMER SHOPPING DNA — CATEGORY VALIDATION")
print("=" * 55)

print("Rows:", dna_rows)
print("Customers:", dna_customers)
print("Duplicate customers:", duplicate_customers)


assert dna_rows == 131209
assert dna_customers == 131209
assert duplicate_customers == 0


print()
print("Category signature validation passed.")


# ============================================================
# 3.10 Preview
# ============================================================

display(
    customer_dna_enriched_df
    .select(
        "user_id",

        "shopping_persona",

        "reorder_loyalty_pct",
        "shopping_frequency",
        "basket_momentum",

        "favorite_department",
        "favorite_department_affinity_pct",

        "second_department",
        "second_department_affinity_pct",

        "third_department",
        "third_department_affinity_pct",

        "favorite_aisle",
        "favorite_aisle_affinity_pct",

        "preferred_time_period",
    )
    .orderBy("user_id")
    .limit(30)
)

Category reference tables loaded.

CUSTOMER SHOPPING DNA — CATEGORY VALIDATION
Rows: 131209
Customers: 131209
Duplicate customers: 0

Category signature validation passed.


user_id,shopping_persona,reorder_loyalty_pct,shopping_frequency,basket_momentum,favorite_department,favorite_department_affinity_pct,second_department,second_department_affinity_pct,third_department,third_department_affinity_pct,favorite_aisle,favorite_aisle_affinity_pct,preferred_time_period
1,GROWING BASKET SHOPPER,69.5,OCCASIONAL,GROWING,snacks,37.3,beverages,22.0,dairy eggs,22.0,soft drinks,22.0,MORNING
2,BALANCED SHOPPER,47.7,OCCASIONAL,STABLE,dairy eggs,24.6,snacks,21.5,produce,18.5,yogurt,21.5,MORNING
5,BALANCED SHOPPER,37.8,REGULAR,STABLE,produce,51.3,dairy eggs,21.6,frozen,5.4,packaged vegetables fruits,21.6,EVENING
7,BALANCED SHOPPER,67.0,REGULAR,STABLE,produce,27.7,beverages,24.8,dairy eggs,15.5,refrigerated,13.1,MORNING
8,BALANCED SHOPPER,26.5,OCCASIONAL,STABLE,produce,55.1,dairy eggs,24.5,canned goods,10.2,fresh vegetables,40.8,LATE NIGHT
9,CURIOUS EXPLORER,23.7,OCCASIONAL,STABLE,dairy eggs,31.6,snacks,17.1,beverages,10.5,yogurt,25.0,AFTERNOON
10,CURIOUS EXPLORER,34.3,OCCASIONAL,GROWING,produce,50.3,pantry,15.4,dairy eggs,11.2,fresh vegetables,19.6,AFTERNOON
13,GROWING BASKET SHOPPER,64.2,REGULAR,GROWING,dairy eggs,33.3,produce,21.0,bakery,17.3,fresh vegetables,16.1,AFTERNOON
14,CURIOUS EXPLORER,32.4,OCCASIONAL,SHRINKING,pantry,24.3,frozen,16.7,produce,15.2,fresh vegetables,11.0,MORNING
17,LOYAL REPLENISHER,71.8,REGULAR,GROWING,beverages,20.1,frozen,17.0,dairy eggs,15.0,water seltzer sparkling water,12.2,AFTERNOON


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 4. CUSTOMER BEHAVIORAL DNA FINGERPRINT
#
# Five interpretable dimensions:
#
# 1. Loyalty
# 2. Exploration
# 3. Routine
# 4. Basket Intensity
# 5. Category Concentration
#
# Every score is expressed on a 0–100 scale.
# ============================================================


# ============================================================
# 4.1 Recover customer-level behavioral variables
#
# These variables are repeated for each customer-product pair
# in the feature table, so collapse them to one row/customer.
# ============================================================

customer_behavior_features_df = (
    reorder_features_df
    .groupBy("user_id")
    .agg(

        F.max(
            "customer_reorder_rate"
        ).alias(
            "dna_customer_reorder_rate"
        ),

        F.max(
            "customer_total_products"
        ).alias(
            "dna_total_product_purchases"
        ),

        F.max(
            "customer_unique_products"
        ).alias(
            "dna_unique_products"
        ),

        F.max(
            "customer_avg_basket_size"
        ).alias(
            "dna_avg_basket_size"
        ),

        F.max(
            "customer_avg_days_between_orders"
        ).alias(
            "dna_avg_days_between_orders"
        ),

        F.max(
            "customer_std_days_between_orders"
        ).alias(
            "dna_std_days_between_orders"
        ),

        F.max(
            "customer_preferred_day_share"
        ).alias(
            "dna_preferred_day_share"
        ),

        F.max(
            "customer_preferred_hour_share"
        ).alias(
            "dna_preferred_hour_share"
        ),
    )
)


# ============================================================
# 4.2 Join behavioral variables into enriched DNA
# ============================================================

customer_dna_scoring_df = (
    customer_dna_enriched_df
    .join(
        customer_behavior_features_df,
        on="user_id",
        how="left",
    )
)


# ============================================================
# Helper: keep every score between 0 and 100
# ============================================================

def clamp_score(column):
    return F.round(
        F.greatest(
            F.lit(0.0),
            F.least(
                F.lit(100.0),
                column,
            ),
        ),
        1,
    )


# ============================================================
# 4.3 LOYALTY SCORE
#
# Measures how strongly the customer tends to repurchase
# products already purchased before.
#
# High = replenishment-oriented
# Low  = more exploratory
# ============================================================

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "loyalty_score",
        clamp_score(
            F.col(
                "dna_customer_reorder_rate"
            ) * 100
        ),
    )
)


# ============================================================
# 4.4 EXPLORATION SCORE
#
# Product diversity:
#
# unique products / total historical product purchases
#
# A customer who constantly buys different products receives
# a higher exploration score.
# ============================================================

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "exploration_score",
        clamp_score(
            F.when(
                F.col(
                    "dna_total_product_purchases"
                ) > 0,
                (
                    F.col(
                        "dna_unique_products"
                    )
                    /
                    F.col(
                        "dna_total_product_purchases"
                    )
                ) * 100,
            )
            .otherwise(
                F.lit(0.0)
            )
        ),
    )
)


# ============================================================
# 4.5 ROUTINE SCORE
#
# Combines three dimensions:
#
# 50% ordering-interval consistency
# 25% preferred-day concentration
# 25% preferred-hour concentration
#
# Example:
# a customer ordering at similar intervals and often on the
# same day/time receives a strong routine score.
# ============================================================


# ------------------------------------------------------------
# Timing consistency
#
# Coefficient of variation:
#
# std days / average days
#
# Lower variation = stronger routine.
# ------------------------------------------------------------

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "timing_consistency_score",
        clamp_score(
            F.when(
                F.col(
                    "dna_avg_days_between_orders"
                ) > 0,
                (
                    1
                    -
                    F.least(
                        F.lit(1.0),
                        (
                            F.col(
                                "dna_std_days_between_orders"
                            )
                            /
                            F.col(
                                "dna_avg_days_between_orders"
                            )
                        ),
                    )
                ) * 100,
            )
            .otherwise(
                F.lit(50.0)
            )
        ),
    )
)


# ------------------------------------------------------------
# Preferred day consistency
# ------------------------------------------------------------

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "day_consistency_score",
        clamp_score(
            F.coalesce(
                F.col(
                    "dna_preferred_day_share"
                ),
                F.lit(0.0),
            ) * 100
        ),
    )
)


# ------------------------------------------------------------
# Preferred hour consistency
# ------------------------------------------------------------

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "hour_consistency_score",
        clamp_score(
            F.coalesce(
                F.col(
                    "dna_preferred_hour_share"
                ),
                F.lit(0.0),
            ) * 100
        ),
    )
)


# ------------------------------------------------------------
# Overall routine
# ------------------------------------------------------------

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "routine_score",
        clamp_score(
            (
                F.col(
                    "timing_consistency_score"
                ) * 0.50
            )
            +
            (
                F.col(
                    "day_consistency_score"
                ) * 0.25
            )
            +
            (
                F.col(
                    "hour_consistency_score"
                ) * 0.25
            )
        ),
    )
)


# ============================================================
# 4.6 BASKET INTENSITY SCORE
#
# Basket size varies substantially between customers.
#
# Instead of choosing an arbitrary basket-size threshold,
# position each customer relative to the whole population.
#
# Percentile rank -> 0–100.
# ============================================================

basket_window = (
    Window
    .orderBy(
        F.col(
            "dna_avg_basket_size"
        )
    )
)


customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "basket_intensity_score",
        F.round(
            F.percent_rank()
            .over(
                basket_window
            ) * 100,
            1,
        ),
    )
)


# ============================================================
# 4.7 CATEGORY CONCENTRATION SCORE
#
# The share of historical purchases belonging to the
# customer's favorite department.
#
# High score:
# shopping is concentrated around a core category.
#
# Low score:
# spending is distributed across many departments.
# ============================================================

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "category_concentration_score",
        clamp_score(
            F.coalesce(
                F.col(
                    "favorite_department_affinity_pct"
                ),
                F.lit(0.0),
            )
        ),
    )
)


# ============================================================
# 4.8 DOMINANT BEHAVIORAL TRAIT
#
# Identify the strongest dimension of the customer's DNA.
# ============================================================

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "dominant_dna_score",
        F.greatest(
            F.col("loyalty_score"),
            F.col("exploration_score"),
            F.col("routine_score"),
            F.col("basket_intensity_score"),
            F.col("category_concentration_score"),
        ),
    )
    .withColumn(
        "dominant_dna_trait",

        F.when(
            F.col("dominant_dna_score")
            ==
            F.col("loyalty_score"),
            "LOYALTY",
        )

        .when(
            F.col("dominant_dna_score")
            ==
            F.col("exploration_score"),
            "EXPLORATION",
        )

        .when(
            F.col("dominant_dna_score")
            ==
            F.col("routine_score"),
            "ROUTINE",
        )

        .when(
            F.col("dominant_dna_score")
            ==
            F.col("basket_intensity_score"),
            "BASKET INTENSITY",
        )

        .otherwise(
            "CATEGORY FOCUS"
        ),
    )
)


# ============================================================
# 4.9 Create compact DNA identifier
#
# Example:
#
# L55 · E31 · R64 · B47 · C29
#
# This will look excellent in the frontend.
# ============================================================

customer_dna_scoring_df = (
    customer_dna_scoring_df
    .withColumn(
        "shopping_dna_code",
        F.concat(
            F.lit("L"),
            F.round(
                F.col("loyalty_score")
            ).cast("int"),

            F.lit(" · E"),
            F.round(
                F.col("exploration_score")
            ).cast("int"),

            F.lit(" · R"),
            F.round(
                F.col("routine_score")
            ).cast("int"),

            F.lit(" · B"),
            F.round(
                F.col("basket_intensity_score")
            ).cast("int"),

            F.lit(" · C"),
            F.round(
                F.col(
                    "category_concentration_score"
                )
            ).cast("int"),
        ),
    )
)


# ============================================================
# 4.10 Final Customer Shopping DNA dataset
# ============================================================

customer_shopping_dna_df = (
    customer_dna_scoring_df
    .select(

        # Identity
        "user_id",

        # Persona
        "shopping_persona",
        "loyalty_profile",
        "shopping_frequency",
        "basket_momentum",
        "shopping_regularity",

        # Behavioral fingerprint
        "loyalty_score",
        "exploration_score",
        "routine_score",
        "basket_intensity_score",
        "category_concentration_score",

        "dominant_dna_trait",
        "dominant_dna_score",
        "shopping_dna_code",

        # Shopping rhythm
        "avg_basket_size",
        "avg_days_between_orders",
        "preferred_hour",
        "preferred_time_period",

        # History
        "customer_prior_orders",
        "customer_unique_products",

        # Category signature
        "favorite_department",
        "favorite_department_affinity_pct",

        "second_department",
        "second_department_affinity_pct",

        "third_department",
        "third_department_affinity_pct",

        "favorite_aisle",
        "favorite_aisle_affinity_pct",

        "second_aisle",
        "second_aisle_affinity_pct",

        "third_aisle",
        "third_aisle_affinity_pct",
    )
)


# ============================================================
# 4.11 VALIDATION
# ============================================================

dna_rows = (
    customer_shopping_dna_df
    .count()
)

dna_customers = (
    customer_shopping_dna_df
    .select("user_id")
    .distinct()
    .count()
)

duplicate_customers = (
    customer_shopping_dna_df
    .groupBy("user_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


score_columns = [
    "loyalty_score",
    "exploration_score",
    "routine_score",
    "basket_intensity_score",
    "category_concentration_score",
]


invalid_score_condition = None

for column in score_columns:

    condition = (
        F.col(column).isNull()
        |
        (F.col(column) < 0)
        |
        (F.col(column) > 100)
    )

    if invalid_score_condition is None:
        invalid_score_condition = condition
    else:
        invalid_score_condition = (
            invalid_score_condition
            |
            condition
        )


invalid_scores = (
    customer_shopping_dna_df
    .filter(
        invalid_score_condition
    )
    .count()
)


print()
print("CUSTOMER SHOPPING DNA — FINGERPRINT VALIDATION")
print("=" * 60)

print("Rows:", dna_rows)
print("Customers:", dna_customers)
print("Duplicate customers:", duplicate_customers)
print("Invalid DNA scores:", invalid_scores)


assert dna_rows == 131209
assert dna_customers == 131209
assert duplicate_customers == 0
assert invalid_scores == 0


print()
print("Behavioral DNA fingerprint validation passed.")


# ============================================================
# 4.12 Preview
# ============================================================

display(
    customer_shopping_dna_df
    .select(
        "user_id",

        "shopping_persona",

        "loyalty_score",
        "exploration_score",
        "routine_score",
        "basket_intensity_score",
        "category_concentration_score",

        "dominant_dna_trait",
        "shopping_dna_code",

        "favorite_department",
        "favorite_aisle",

        "shopping_frequency",
        "preferred_time_period",
    )
    .orderBy("user_id")
    .limit(30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



CUSTOMER SHOPPING DNA — FINGERPRINT VALIDATION
Rows: 131209
Customers: 131209
Duplicate customers: 0
Invalid DNA scores: 0

Behavioral DNA fingerprint validation passed.


user_id,shopping_persona,loyalty_score,exploration_score,routine_score,basket_intensity_score,category_concentration_score,dominant_dna_trait,shopping_dna_code,favorite_department,favorite_aisle,shopping_frequency,preferred_time_period
1,GROWING BASKET SHOPPER,69.5,30.5,42.4,26.0,37.3,LOYALTY,L70 · E31 · R42 · B26 · C37,snacks,soft drinks,OCCASIONAL,MORNING
2,BALANCED SHOPPER,47.7,52.3,36.8,79.1,24.6,BASKET INTENSITY,L48 · E52 · R37 · B79 · C25,dairy eggs,yogurt,OCCASIONAL,MORNING
5,BALANCED SHOPPER,37.8,62.2,59.9,52.3,51.3,EXPLORATION,L38 · E62 · R60 · B52 · C51,produce,packaged vegetables fruits,REGULAR,EVENING
7,BALANCED SHOPPER,67.0,33.0,20.5,59.8,27.7,LOYALTY,L67 · E33 · R21 · B60 · C28,produce,refrigerated,REGULAR,MORNING
8,BALANCED SHOPPER,26.5,73.5,75.0,86.9,55.1,BASKET INTENSITY,L27 · E74 · R75 · B87 · C55,produce,fresh vegetables,OCCASIONAL,LATE NIGHT
9,CURIOUS EXPLORER,23.7,76.3,33.3,98.0,31.6,BASKET INTENSITY,L24 · E76 · R33 · B98 · C32,dairy eggs,yogurt,OCCASIONAL,AFTERNOON
10,CURIOUS EXPLORER,34.3,65.7,51.7,99.0,50.3,BASKET INTENSITY,L34 · E66 · R52 · B99 · C50,produce,fresh vegetables,OCCASIONAL,AFTERNOON
13,GROWING BASKET SHOPPER,64.2,35.8,54.1,32.7,33.3,LOYALTY,L64 · E36 · R54 · B33 · C33,dairy eggs,fresh vegetables,REGULAR,AFTERNOON
14,CURIOUS EXPLORER,32.4,67.6,47.0,86.5,24.3,BASKET INTENSITY,L32 · E68 · R47 · B87 · C24,pantry,fresh vegetables,OCCASIONAL,MORNING
17,LOYAL REPLENISHER,71.8,28.2,13.5,37.7,20.1,LOYALTY,L72 · E28 · R14 · B38 · C20,beverages,water seltzer sparkling water,REGULAR,AFTERNOON


In [0]:
from pyspark.sql import functions as F


# ============================================================
# 5. HUMAN-READABLE SHOPPING DNA INTELLIGENCE
#
# Convert the behavioral fingerprint into concise,
# deterministic client-facing explanations.
# ============================================================


# ============================================================
# 5.1 Persona description
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_df

    .withColumn(
        "persona_description",

        F.when(
            F.col("shopping_persona") == "LOYAL REPLENISHER",
            F.lit(
                "You frequently return to familiar products "
                "and maintain a repeat-oriented shopping pattern."
            ),
        )

        .when(
            F.col("shopping_persona") == "CURIOUS EXPLORER",
            F.lit(
                "You explore a broad variety of products "
                "and regularly diversify what enters your basket."
            ),
        )

        .when(
            F.col("shopping_persona") == "FREQUENT PLANNER",
            F.lit(
                "You shop frequently and follow a relatively "
                "active purchasing rhythm."
            ),
        )

        .when(
            F.col("shopping_persona") == "GROWING BASKET SHOPPER",
            F.lit(
                "Your recent baskets are becoming larger than "
                "your historical shopping pattern."
            ),
        )

        .otherwise(
            F.lit(
                "Your shopping behavior combines repeat purchases, "
                "product variety and a balanced purchasing rhythm."
            ),
        ),
    )
)


# ============================================================
# 5.2 Loyalty interpretation
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "loyalty_insight",

        F.when(
            F.col("loyalty_profile") == "HIGH LOYALTY",
            F.concat(
                F.lit("Repeat behavior is strong: "),
                F.col("loyalty_score").cast("string"),
                F.lit(
                    "% of the loyalty signal points toward "
                    "familiar-product purchasing."
                ),
            ),
        )

        .when(
            F.col("loyalty_profile") == "MODERATE LOYALTY",
            F.concat(
                F.lit(
                    "You balance familiar products with variety, "
                    "with a loyalty score of "
                ),
                F.col("loyalty_score").cast("string"),
                F.lit("%."),
            ),
        )

        .otherwise(
            F.concat(
                F.lit(
                    "Your behavior is relatively exploratory, "
                    "with a loyalty score of "
                ),
                F.col("loyalty_score").cast("string"),
                F.lit("%."),
            ),
        ),
    )
)


# ============================================================
# 5.3 Shopping cadence interpretation
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "frequency_insight",

        F.when(
            F.col("shopping_frequency") == "FREQUENT",
            F.concat(
                F.lit("You shop frequently, approximately every "),
                F.col("avg_days_between_orders").cast("string"),
                F.lit(" days."),
            ),
        )

        .when(
            F.col("shopping_frequency") == "REGULAR",
            F.concat(
                F.lit("You maintain a regular shopping cycle of about "),
                F.col("avg_days_between_orders").cast("string"),
                F.lit(" days between orders."),
            ),
        )

        .otherwise(
            F.concat(
                F.lit(
                    "Your orders are more widely spaced, averaging "
                ),
                F.col("avg_days_between_orders").cast("string"),
                F.lit(" days apart."),
            ),
        ),
    )
)


# ============================================================
# 5.4 Basket momentum interpretation
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "basket_insight",

        F.when(
            F.col("basket_momentum") == "GROWING",
            F.concat(
                F.lit(
                    "Your recent basket is trending upward from "
                    "your historical average of "
                ),
                F.col("avg_basket_size").cast("string"),
                F.lit(" items."),
            ),
        )

        .when(
            F.col("basket_momentum") == "SHRINKING",
            F.concat(
                F.lit(
                    "Your recent basket size is trending below "
                    "your historical average of "
                ),
                F.col("avg_basket_size").cast("string"),
                F.lit(" items."),
            ),
        )

        .otherwise(
            F.concat(
                F.lit(
                    "Your basket size is relatively stable around "
                ),
                F.col("avg_basket_size").cast("string"),
                F.lit(" items."),
            ),
        ),
    )
)


# ============================================================
# 5.5 Category preference interpretation
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "category_insight",

        F.when(
            F.col("favorite_department").isNotNull(),

            F.concat(
                F.lit("Your strongest department is "),
                F.initcap(
                    F.col("favorite_department")
                ),
                F.lit(" at "),
                F.col(
                    "favorite_department_affinity_pct"
                ).cast("string"),
                F.lit(
                    "% of your historical department affinity"
                ),

                F.when(
                    F.col("favorite_aisle").isNotNull(),
                    F.concat(
                        F.lit(", with "),
                        F.initcap(
                            F.col("favorite_aisle")
                        ),
                        F.lit(
                            " as your strongest aisle."
                        ),
                    ),
                )
                .otherwise(
                    F.lit(".")
                ),
            ),
        )

        .otherwise(
            F.lit(
                "No dominant shopping category has been identified."
            ),
        ),
    )
)


# ============================================================
# 5.6 Preferred shopping moment
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "preferred_hour_label",

        F.format_string(
            "%02d:00",
            F.col("preferred_hour"),
        ),
    )

    .withColumn(
        "time_insight",

        F.concat(
            F.lit("Your strongest shopping-time signal is "),
            F.initcap(
                F.lower(
                    F.col("preferred_time_period")
                )
            ),
            F.lit(
                ", with a preferred hour around "
            ),
            F.col("preferred_hour_label"),
            F.lit("."),
        ),
    )
)


# ============================================================
# 5.7 Dominant trait explanation
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "dominant_trait_insight",

        F.when(
            F.col("dominant_dna_trait") == "LOYALTY",
            F.lit(
                "Your strongest behavioral signal is loyalty "
                "to previously purchased products."
            ),
        )

        .when(
            F.col("dominant_dna_trait") == "EXPLORATION",
            F.lit(
                "Your strongest behavioral signal is product exploration "
                "and variety."
            ),
        )

        .when(
            F.col("dominant_dna_trait") == "ROUTINE",
            F.lit(
                "Your strongest behavioral signal is shopping routine "
                "and timing consistency."
            ),
        )

        .when(
            F.col("dominant_dna_trait") == "BASKET INTENSITY",
            F.lit(
                "Your strongest behavioral signal is the relative size "
                "of your shopping baskets."
            ),
        )

        .otherwise(
            F.lit(
                "Your strongest behavioral signal is concentration "
                "around preferred shopping categories."
            ),
        ),
    )
)


# ============================================================
# 5.8 Compact narrative for the frontend
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "shopping_dna_summary",

        F.concat_ws(
            " ",

            F.col("persona_description"),

            F.col("frequency_insight"),

            F.col("basket_insight"),

            F.when(
                F.col("favorite_department").isNotNull(),
                F.concat(
                    F.lit("Your strongest category is "),
                    F.initcap(
                        F.col("favorite_department")
                    ),
                    F.lit("."),
                ),
            ),

            F.col("dominant_trait_insight"),
        ),
    )
)


# ============================================================
# 5.9 Client-facing headline
# ============================================================

customer_shopping_dna_narrative_df = (
    customer_shopping_dna_narrative_df

    .withColumn(
        "dna_headline",

        F.concat(
            F.initcap(
                F.lower(
                    F.col("shopping_persona")
                )
            ),
            F.lit(" · "),
            F.initcap(
                F.lower(
                    F.col("dominant_dna_trait")
                )
            ),
        ),
    )
)


# ============================================================
# 5.10 Final serving dataframe
# ============================================================

customer_shopping_dna_serving_df = (
    customer_shopping_dna_narrative_df

    .select(

        # ----------------------------------------------------
        # Identity
        # ----------------------------------------------------

        "user_id",

        "shopping_persona",
        "dna_headline",
        "shopping_dna_code",

        # ----------------------------------------------------
        # Main narrative
        # ----------------------------------------------------

        "shopping_dna_summary",
        "persona_description",
        "dominant_trait_insight",

        # ----------------------------------------------------
        # DNA fingerprint
        # ----------------------------------------------------

        "loyalty_score",
        "exploration_score",
        "routine_score",
        "basket_intensity_score",
        "category_concentration_score",

        "dominant_dna_trait",
        "dominant_dna_score",

        # ----------------------------------------------------
        # Shopping behavior
        # ----------------------------------------------------

        "loyalty_profile",
        "shopping_frequency",
        "basket_momentum",
        "shopping_regularity",

        "avg_basket_size",
        "avg_days_between_orders",

        "preferred_hour",
        "preferred_hour_label",
        "preferred_time_period",

        # ----------------------------------------------------
        # Customer history
        # ----------------------------------------------------

        "customer_prior_orders",
        "customer_unique_products",

        # ----------------------------------------------------
        # Category signature
        # ----------------------------------------------------

        "favorite_department",
        "favorite_department_affinity_pct",

        "second_department",
        "second_department_affinity_pct",

        "third_department",
        "third_department_affinity_pct",

        "favorite_aisle",
        "favorite_aisle_affinity_pct",

        "second_aisle",
        "second_aisle_affinity_pct",

        "third_aisle",
        "third_aisle_affinity_pct",

        # ----------------------------------------------------
        # Detailed explainability
        # ----------------------------------------------------

        "loyalty_insight",
        "frequency_insight",
        "basket_insight",
        "category_insight",
        "time_insight",
    )
)


# ============================================================
# 5.11 Validation
# ============================================================

row_count = customer_shopping_dna_serving_df.count()

customer_count = (
    customer_shopping_dna_serving_df
    .select("user_id")
    .distinct()
    .count()
)

duplicate_count = (
    customer_shopping_dna_serving_df
    .groupBy("user_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

missing_summary_count = (
    customer_shopping_dna_serving_df
    .filter(
        F.col("shopping_dna_summary").isNull()
        |
        (
            F.length(
                F.trim(
                    F.col("shopping_dna_summary")
                )
            ) == 0
        )
    )
    .count()
)


print()
print("CUSTOMER SHOPPING DNA — NARRATIVE VALIDATION")
print("=" * 60)

print("Rows:", row_count)
print("Customers:", customer_count)
print("Duplicate customers:", duplicate_count)
print("Missing summaries:", missing_summary_count)


assert row_count == 131209
assert customer_count == 131209
assert duplicate_count == 0
assert missing_summary_count == 0


print()
print("Shopping DNA narrative validation passed.")


# ============================================================
# 5.12 Preview
# ============================================================

display(
    customer_shopping_dna_serving_df

    .select(
        "user_id",
        "dna_headline",
        "shopping_dna_code",
        "shopping_dna_summary",
        "dominant_dna_trait",
        "favorite_department",
        "favorite_aisle",
        "preferred_time_period",
    )

    .orderBy("user_id")

    .limit(30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



CUSTOMER SHOPPING DNA — NARRATIVE VALIDATION
Rows: 131209
Customers: 131209
Duplicate customers: 0
Missing summaries: 0

Shopping DNA narrative validation passed.


user_id,dna_headline,shopping_dna_code,shopping_dna_summary,dominant_dna_trait,favorite_department,favorite_aisle,preferred_time_period
1,Growing Basket Shopper · Loyalty,L70 · E31 · R42 · B26 · C37,"Your recent baskets are becoming larger than your historical shopping pattern. Your orders are more widely spaced, averaging 19.6 days apart. Your recent basket is trending upward from your historical average of 5.9 items. Your strongest category is Snacks. Your strongest behavioral signal is loyalty to previously purchased products.",LOYALTY,snacks,soft drinks,MORNING
2,Balanced Shopper · Basket Intensity,L48 · E52 · R37 · B79 · C25,"Your shopping behavior combines repeat purchases, product variety and a balanced purchasing rhythm. Your orders are more widely spaced, averaging 15.2 days apart. Your basket size is relatively stable around 13.9 items. Your strongest category is Dairy Eggs. Your strongest behavioral signal is the relative size of your shopping baskets.",BASKET INTENSITY,dairy eggs,yogurt,MORNING
5,Balanced Shopper · Exploration,L38 · E62 · R60 · B52 · C51,"Your shopping behavior combines repeat purchases, product variety and a balanced purchasing rhythm. You maintain a regular shopping cycle of about 13.3 days between orders. Your basket size is relatively stable around 9.3 items. Your strongest category is Produce. Your strongest behavioral signal is product exploration and variety.",EXPLORATION,produce,packaged vegetables fruits,EVENING
7,Balanced Shopper · Loyalty,L67 · E33 · R21 · B60 · C28,"Your shopping behavior combines repeat purchases, product variety and a balanced purchasing rhythm. You maintain a regular shopping cycle of about 10.7 days between orders. Your basket size is relatively stable around 10.3 items. Your strongest category is Produce. Your strongest behavioral signal is loyalty to previously purchased products.",LOYALTY,produce,refrigerated,MORNING
8,Balanced Shopper · Basket Intensity,L27 · E74 · R75 · B87 · C55,"Your shopping behavior combines repeat purchases, product variety and a balanced purchasing rhythm. Your orders are more widely spaced, averaging 30.0 days apart. Your basket size is relatively stable around 16.3 items. Your strongest category is Produce. Your strongest behavioral signal is the relative size of your shopping baskets.",BASKET INTENSITY,produce,fresh vegetables,LATE NIGHT
9,Curious Explorer · Basket Intensity,L24 · E76 · R33 · B98 · C32,"You explore a broad variety of products and regularly diversify what enters your basket. Your orders are more widely spaced, averaging 18.0 days apart. Your basket size is relatively stable around 25.3 items. Your strongest category is Dairy Eggs. Your strongest behavioral signal is the relative size of your shopping baskets.",BASKET INTENSITY,dairy eggs,yogurt,AFTERNOON
10,Curious Explorer · Basket Intensity,L34 · E66 · R52 · B99 · C50,"You explore a broad variety of products and regularly diversify what enters your basket. Your orders are more widely spaced, averaging 19.8 days apart. Your recent basket is trending upward from your historical average of 28.6 items. Your strongest category is Produce. Your strongest behavioral signal is the relative size of your shopping baskets.",BASKET INTENSITY,produce,fresh vegetables,AFTERNOON
13,Growing Basket Shopper · Loyalty,L64 · E36 · R54 · B33 · C33,Your recent baskets are becoming larger than your historical shopping pattern. You maintain a regular shopping cycle of about 7.6 days between orders. Your recent basket is trending upward from your historical average of 6.8 items. Your strongest category is Dairy Eggs. Your strongest behavioral signal is loyalty to previously purchased products.,LOYALTY,dairy eggs,fresh vegetables,AFTERNOON
14,Curious Explorer · Basket Intensity,L32 · E68 · R47 · B87 · C24,"You explore a broad variety of products and regularly diversify what enters your basket. Your orders are more widely spaced, averaging 22.1 days apart. Your recent basket size is trending be

In [0]:
# ============================================================
# 6. Neon PostgreSQL connection
# ============================================================

neon_host = (
    "ep-summer-sea-ag9wnzfy-pooler."
    "c-2.eu-central-1.aws.neon.tech"
)

neon_port = "5432"
neon_database = "neondb"
neon_user = "neondb_owner"

# Password remains protected in Databricks Secrets
neon_password = dbutils.secrets.get(
    scope="neon",
    key="password",
)

print("Neon configuration loaded successfully.")
print("Host:", neon_host)
print("Database:", neon_database)
print("User:", neon_user)

Neon configuration loaded successfully.
Host: ep-summer-sea-ag9wnzfy-pooler.c-2.eu-central-1.aws.neon.tech
Database: neondb
User: neondb_owner


In [0]:
# ============================================================
# 7. Export Customer Shopping DNA to Neon PostgreSQL
# ============================================================

target_table = "customer_shopping_dna"


# ------------------------------------------------------------
# Export final serving dataset
# ------------------------------------------------------------

(
    customer_shopping_dna_serving_df
    .write
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", target_table)
    .option("user", neon_user)
    .option("password", neon_password)
    .option("batchsize", "5000")
    .option("numPartitions", "2")
    .mode("overwrite")
    .save()
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

exported_rows = customer_shopping_dna_serving_df.count()

print()
print("CUSTOMER SHOPPING DNA EXPORT")
print("=" * 50)

print("Export completed:", target_table)
print("Rows exported:", exported_rows)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



CUSTOMER SHOPPING DNA EXPORT
Export completed: customer_shopping_dna
Rows exported: 131209


In [0]:
# ============================================================
# 8. Verify Customer Shopping DNA stored in Neon PostgreSQL
# ============================================================

neon_dna_df = (
    spark.read
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", "customer_shopping_dna")
    .option("user", neon_user)
    .option("password", neon_password)
    .load()
)


# ============================================================
# 8.1 Row and customer counts
# ============================================================

neon_row_count = neon_dna_df.count()

neon_customer_count = (
    neon_dna_df
    .select("user_id")
    .distinct()
    .count()
)


# ============================================================
# 8.2 Duplicate customer check
# ============================================================

duplicate_customer_count = (
    neon_dna_df
    .groupBy("user_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


# ============================================================
# 8.3 Required-field validation
# ============================================================

required_columns = [
    "user_id",
    "shopping_persona",
    "dna_headline",
    "shopping_dna_code",
    "shopping_dna_summary",
    "loyalty_score",
    "exploration_score",
    "routine_score",
    "basket_intensity_score",
    "category_concentration_score",
    "dominant_dna_trait",
]


missing_required_values = (
    neon_dna_df
    .filter(
        F.col("user_id").isNull()
        |
        F.col("shopping_persona").isNull()
        |
        F.col("dna_headline").isNull()
        |
        F.col("shopping_dna_code").isNull()
        |
        F.col("shopping_dna_summary").isNull()
        |
        F.col("loyalty_score").isNull()
        |
        F.col("exploration_score").isNull()
        |
        F.col("routine_score").isNull()
        |
        F.col("basket_intensity_score").isNull()
        |
        F.col("category_concentration_score").isNull()
        |
        F.col("dominant_dna_trait").isNull()
    )
    .count()
)


# ============================================================
# 8.4 Validate score ranges
# ============================================================

invalid_score_count = (
    neon_dna_df
    .filter(
        (F.col("loyalty_score") < 0)
        |
        (F.col("loyalty_score") > 100)
        |
        (F.col("exploration_score") < 0)
        |
        (F.col("exploration_score") > 100)
        |
        (F.col("routine_score") < 0)
        |
        (F.col("routine_score") > 100)
        |
        (F.col("basket_intensity_score") < 0)
        |
        (F.col("basket_intensity_score") > 100)
        |
        (F.col("category_concentration_score") < 0)
        |
        (F.col("category_concentration_score") > 100)
    )
    .count()
)


# ============================================================
# 8.5 Validation output
# ============================================================

print()
print("NEON CUSTOMER SHOPPING DNA VALIDATION")
print("=" * 60)

print("Rows:", neon_row_count)
print("Customers:", neon_customer_count)
print("Duplicate customers:", duplicate_customer_count)
print("Missing required values:", missing_required_values)
print("Invalid DNA scores:", invalid_score_count)


assert neon_row_count == 131209
assert neon_customer_count == 131209
assert duplicate_customer_count == 0
assert missing_required_values == 0
assert invalid_score_count == 0


print()
print("Neon Shopping DNA validation passed.")


NEON CUSTOMER SHOPPING DNA VALIDATION
Rows: 131209
Customers: 131209
Duplicate customers: 0
Missing required values: 0
Invalid DNA scores: 0

Neon Shopping DNA validation passed.
